# Fig. 2: convergence and runtime

Run cells in order from a fresh kernel. Settings are in `configs/experiments.json`; artifacts use the paths described in the README. Timings depend on hardware and software.


In [ ]:
from fthbf.runtime import (DATA_DIR, CHECKPOINT_DIR, checkpoint_path, load_artifact,
                           result_dir, read_config, seed_everything, record_run)
import torch
cfg = read_config("experiments.json")["Fig00"]
seed_everything(cfg["seed"])
record_run("Fig00", cfg)


In [ ]:
# ============================================================
# This script:
# (i)  plots the convergence behavior of classical WMMSE
# (ii) measures average runtime for different iteration counts
# ============================================================

import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import trange
from solutions.trihybrid_WMMSE import TriHybridWMMSE
import os

# ============================================================
# Basic settings
# ============================================================

BASE_SEED = cfg["seed"]
cdtype = torch.cfloat

K = cfg["users"]
I_plot = cfg["classical_plot_iterations"]                    # for convergence curve
I_list = cfg["classical_runtime_iterations"]         # for runtime measurement
N_trials = cfg["runtime_trials"]                   # number of repetitions

torch.manual_seed(BASE_SEED)

# ============================================================
# Load system parameters
# ============================================================

SAVE_DIR = DATA_DIR
parameters = load_artifact(os.path.join(SAVE_DIR, "parameters.pt"))
Nt = parameters["Nt"]
Nc = parameters["Nc"]
BW = parameters["BW"]
P = parameters["P"]
Mk = parameters["Mk"]
eta = parameters["eta"]
Ns_k = parameters["Ns_k"]
sigma2 = parameters["sigma2"]

Ns = Ns_k * K

# ============================================================
# Load and select channels
# ============================================================

channels = load_artifact(os.path.join(SAVE_DIR, "channels_deepMIMO_test.pt"))

sel_idxs = [379, 2065, 8190, 4622]    # strong channel examples
selected_channels = channels[sel_idxs]      # (K, Mk, Nt*Nc, 1)

print("Selected indices:", sel_idxs)
print("Selected channels shape:", selected_channels.shape)

selected_channels_torch = torch.tensor(
    selected_channels, dtype=cdtype
).squeeze(-1)                   # (K, Mk, Nt*Nc)

H_list = [selected_channels_torch[k] for k in range(K)]

# ============================================================
# Helper: initialize u0, v0
# ============================================================

def initialize_uv():
    u0 = [torch.ones(Nc, 1, dtype=cdtype) for _ in range(Nt)]
    v0 = [torch.ones(Ns, 1, dtype=cdtype) for _ in range(Nt)]

    for n in range(Nt):
        norm_vn = torch.linalg.norm(v0[n])
        if norm_vn > torch.sqrt(P):
            v0[n] = v0[n] / norm_vn * torch.sqrt(P)

    return u0, v0

# ============================================================
# (i) Run once for convergence curve (200 iterations)
# ============================================================

print("\nRunning WMMSE for convergence curve...")

u0, v0 = initialize_uv()

solver = TriHybridWMMSE(
    H_list=H_list,
    Ns_k=[Ns_k] * K,
    Mk=[Mk] * K,
    sigma2=[sigma2] * K,
    beta=[1.0] * K,
    P=[P] * Nt,
    eta=eta,
    u_init=u0,
    v_init=v0
)

start_time = time.time()
rate_trace, FA_opt, FD_opt, u_opt, v_opt = solver.run_classical(
    I_max=I_plot, verbose=False
)
end_time = time.time()

print(f"WMMSE (I_max={I_plot}) completed in {end_time - start_time:.2f} seconds.")

# ---- plot convergence curve ----
plt.figure(figsize=(10, 3))
plt.plot(rate_trace, marker='o', linewidth=1)
plt.xticks(range(0, I_plot + 1, 10))
plt.xlabel("Iteration")
plt.ylabel("Sum-rate")
plt.grid(True)
plt.tight_layout()
plt.show()

# ============================================================
# (ii) Runtime statistics with progress bars
# ============================================================

print("\nRuntime statistics (mean ± std):")

time_results = {}

for I_max in I_list:
    times = []

    print(f"\nRunning runtime trials for I_max = {I_max}")

    for trial in trange(
        N_trials,
        desc=f"I_max = {I_max}",
        leave=True
    ):
        torch.manual_seed(BASE_SEED + trial)

        u0, v0 = initialize_uv()

        solver = TriHybridWMMSE(
            H_list=H_list,
            Ns_k=[Ns_k] * K,
            Mk=[Mk] * K,
            sigma2=[sigma2] * K,
            beta=[1.0] * K,
            P=[P] * Nt,
            eta=eta,
            u_init=u0,
            v_init=v0
        )

        start = time.time()
        solver.run_classical(I_max=I_max, verbose=False)
        end = time.time()

        times.append(end - start)

    times = np.array(times)

    time_results[I_max] = {
        "mean": times.mean(),
        "std": times.std()
    }

    print(
        f"I_max = {I_max:3d} | "
        f"mean time = {times.mean():.4f} s | "
        f"std = {times.std():.4f} s"
    )

# ============================================================
# Optional: visualize runtime scaling
# ============================================================

plt.figure(figsize=(5, 3))
plt.errorbar(
    I_list,
    [time_results[I]["mean"] for I in I_list],
    yerr=[time_results[I]["std"] for I in I_list],
    marker='o',
    capsize=4
)
plt.xlabel("Number of WMMSE Iterations")
plt.ylabel("Runtime (seconds)")
plt.grid(True)
plt.tight_layout()
plt.show()


if False:
    print("{%")
    for i, y in enumerate(rate_trace, start=0):  
        print(f"{i}\t{y:.6f}\\\\")
    print("};")


In [ ]:
# This script decomposes the virtual fully-digital precoder obtained from WMMSE into hybrid precoding components using the BFGS-based algorithm. 
# It then checks the constraints of the hybrid precoding components.

from solutions.utils import HybridPrecodingBFGS, computeWSR, checkHybridConstraints

Nrf = cfg["rf_chains"]
solver = HybridPrecodingBFGS(FD_opt, Nrf)
FRF_opt, FBB_opt = solver.solve(max_iter=100, use_B0=True, verbose=False)

R, _ = computeWSR(FA_opt, FRF_opt, FBB_opt, H_list, parameters)
print(f"Sum-rate after hybrid precoding: {R:.4f} bps/Hz")

print("----- Checking hybrid precoding constraints...")
phase_ok, papc_ok, info = checkHybridConstraints(FRF_opt, FBB_opt, P, tol=1e-6, verbose=True)
if not phase_ok:
    print("Warning: Phase shifter constraint violated!")
if not papc_ok:
    FBB_opt = FBB_opt / torch.sqrt(torch.tensor(info["max_papc_ratio"]))
    print("----- Scaling FBB_opt to satisfy Per-DMA input power constraint...")
phase_ok, papc_ok, info = checkHybridConstraints(FRF_opt, FBB_opt, P, tol=1e-6, verbose=True)    

R_final, _ = computeWSR(FA_opt, FRF_opt, FBB_opt, H_list, parameters)
print(f"Final sum-rate after hybrid precoding: {R_final:.4f} bps/Hz")

In [ ]:
# ============================================================
# This script:
# (i)  plots the convergence behavior of the BFGS-based
#      hybrid precoding algorithm on a single channel
#      realization
# (ii) measures the average runtime for different maximum
#      numbers of BFGS iterations
# ============================================================

import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import trange

from solutions.utils import HybridPrecodingBFGS, computeWSR, checkHybridConstraints

# ============================================================
# Basic settings
# ============================================================

Nrf = cfg["rf_chains"]

I_plot = cfg["hybrid_plot_iterations"]                     # for performance curve
I_list = cfg["hybrid_runtime_iterations"]           # for runtime measurement
N_trials = cfg["runtime_trials"]                   # number of repetitions

BASE_SEED = cfg["seed"]
torch.manual_seed(BASE_SEED)

# ============================================================
# (i) Run once: performance curve up to 60 iterations
# ============================================================

print("\nRunning Hybrid BFGS for sum-rate curve...")

R_list = []
time_list = []

for max_iter in range(1, I_plot + 1):

    start_time = time.time()

    # Initialize solver
    solver = HybridPrecodingBFGS(FD_opt, Nrf)

    # Solve hybrid precoding
    FRF_opt, FBB_opt = solver.solve(
        max_iter=max_iter,
        use_B0=True,
        verbose=False
    )

    # Compute sum-rate
    R, _ = computeWSR(FA_opt, FRF_opt, FBB_opt, H_list, parameters)

    # Check constraints
    phase_ok, papc_ok, info = checkHybridConstraints(
        FRF_opt, FBB_opt, P, tol=1e-6, verbose=False
    )

    # Enforce PAPC if violated
    if not papc_ok:
        FBB_opt = FBB_opt / torch.sqrt(
            torch.tensor(info["max_papc_ratio"])
        )

    # Final sum-rate
    R_final, _ = computeWSR(
        FA_opt, FRF_opt, FBB_opt, H_list, parameters
    )

    end_time = time.time()

    R_list.append(R_final.item())
    time_list.append(end_time - start_time)

    print(
        f"iter={max_iter:3d}, "
        f"R_final={R_final:.4f}, "
        f"time={end_time - start_time:.4f} s"
    )

# ---- plot sum-rate curve ----
plt.figure(figsize=(6, 4))
plt.plot(range(1, I_plot + 1), R_list, marker='o')
plt.xlabel("BFGS Iterations")
plt.ylabel("Final Sum Rate (bps/Hz)")
plt.title("Sum Rate vs. BFGS Iterations")
plt.grid(True)
plt.tight_layout()
plt.show()

# ============================================================
# (ii) Runtime statistics with progress bars
# ============================================================

print("\nRuntime statistics (mean ± std):")

time_results = {}

for I_max in I_list:
    times = []

    print(f"\nRunning runtime trials for max_iter = {I_max}")

    for trial in trange(
        N_trials,
        desc=f"max_iter = {I_max}",
        leave=True
    ):
        torch.manual_seed(BASE_SEED + trial)

        start = time.time()

        solver = HybridPrecodingBFGS(FD_opt, Nrf)

        FRF_opt, FBB_opt = solver.solve(
            max_iter=I_max,
            use_B0=True,
            verbose=False
        )

        # Constraint check (same as above, included for fairness)
        phase_ok, papc_ok, info = checkHybridConstraints(
            FRF_opt, FBB_opt, P, tol=1e-6, verbose=False
        )

        if not papc_ok:
            FBB_opt = FBB_opt / torch.sqrt(
                torch.tensor(info["max_papc_ratio"])
            )

        end = time.time()
        times.append(end - start)

    times = np.array(times)

    time_results[I_max] = {
        "mean": times.mean(),
        "std": times.std()
    }

    print(
        f"max_iter = {I_max:3d} | "
        f"mean time = {times.mean():.4f} s | "
        f"std = {times.std():.4f} s"
    )

# ============================================================
# Optional: visualize runtime scaling
# ============================================================

plt.figure(figsize=(5, 3))
plt.errorbar(
    I_list,
    [time_results[I]["mean"] for I in I_list],
    yerr=[time_results[I]["std"] for I in I_list],
    marker='o',
    capsize=4
)
plt.xlabel("BFGS Iterations")
plt.ylabel("Runtime (seconds)")
plt.title("Runtime vs. BFGS Iterations")
plt.grid(True)
plt.tight_layout()
plt.show()

if True:
    print("{%")
    for i, y in enumerate(R_list, start=0):  
        print(f"{i}\t{y:.6f}\\\\")
    print("};")